In [ ]:
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
import string
import random

np.random.seed(30)  # Agar setiap running kode hasilnya angka yang sama setiap membangkitkan angka random

karakter = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'

def vigenere_cipher(text, key):
    enc = karakter
    ciph = ''
    key = key
    text = text
    j = 0
    for i in range(len(text)):
        try:
            if text[i].isalpha() or text[i].isnumeric():
                ciph += enc[(enc.index(text[i])+enc.index(key[j])) % len(enc)]
                j += 1
                j %= len(key)
            else:
                ciph += text[i]
        except:
            ciph += text[i]
    return (ciph, key)

def createDataset(df, algorithm, name):
    """
    Args:
    df(DataFrame): Pandas DataFrame of plaintext files
    algorithm(function): A function of the form algorithm(string,key) that returns a tuple of the form (ciphertext,key)
    name(string): Name of the algorithm without .csv extension. This is what the created dataset will be stored as i.e. name.csv
    """

    keys = 20
    randStrings = [randStr() for i in range(keys)]
    start = time.time()
    
    df['Passwords'] = df['Passwords'].apply(lambda x: str(x))
    print("Applying algorithm to passwords")
    num_data = len(df)
    keys_list = (randStrings * (num_data // len(randStrings) + 1))[:num_data]
    np.random.shuffle(keys_list)
    
    results = [algorithm(pw, key) for pw, key in zip(df['Passwords'], keys_list)]
    results = list(zip(*results))
    ciphertext = list(results[0])
    keys = list(results[1])

    print(f"Generating the dataset and saving the file as data/{name}.csv")
    # Membangkitkan dataset
    df['ciphertext'] = ciphertext
    df['key'] = keys
    df.to_excel(f'{name}.xlsx', index=False)
    end = time.time()
    print(f"Done in {end-start} seconds.")

def randStr(chars=karakter, N=5):
    m = random.randint(1,N)
    return ''.join(random.choice(chars) for _ in range(m))

if __name__ == "__main__":
    tqdm.pandas()  # Progress bar

    # Membaca file
    df = pd.read_excel(
        "filtered_passwords.xlsx",
        names=["Passwords"]
        )

    maxRows = 6000
    df = df.sample(frac=1)[:maxRows]

    createDataset(df, vigenere_cipher, 'Vigenere')